# Generate Qwen2.5 Next-Token Logit Vectors on Colab

This notebook loads one Qwen2.5 instruction model at a time and extracts the next-token logit vector for any context string you provide.

The available models are:

- `Qwen/Qwen2.5-7B-Instruct`
- `Qwen/Qwen2.5-14B-Instruct`
- `Qwen/Qwen2.5-32B-Instruct`

Before running the notebook, choose a GPU in Colab:

`Runtime` -> `Change runtime type` -> `GPU`

## Step 1: Install Dependencies

This installs the Hugging Face loading stack. `bitsandbytes` is needed only if you choose `8bit` or `4bit` loading.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece safetensors

## Step 2: Check Your GPU

This does not change anything. It just shows which GPU Colab gave you.

In [ ]:
!nvidia-smi

## Step 3: Choose and Safely Load One Model

Edit `MODEL_KEY` and `LOAD_MODE`, then run the next cell. The loader automatically cleans up any previously loaded model before loading the new one. To switch models later, change these two variables and rerun only this cell.


In [ ]:
# Optional storage/memory cleanup before loading a model.
# This cell is self-contained so the notebook can run cleanly from a fresh runtime.
import gc

for variable_name in ("model", "tokenizer"):
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception as exc:
    print("Torch cleanup skipped:", repr(exc))

# Clear Hugging Face and Torch caches from the default Colab locations.
# Use this when you intentionally want to free storage before a different model run.
!rm -rf /root/.cache/huggingface
!rm -rf /root/.cache/torch
!rm -rf /content/.cache/huggingface
!rm -rf /content/.cache/torch

# Clear pip/cache/temp junk.
!rm -rf /root/.cache/pip
!rm -rf /tmp/*

# Optional: remove local model output/logit files only after downloading them.
# !rm -f /content/*.pt
# !rm -f /content/*.zip

# Show remaining disk use.
!du -h --max-depth=2 /root 2>/dev/null | sort -hr | head -20
!du -h --max-depth=2 /content 2>/dev/null | sort -hr | head -20
!df -h


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_KEY = "7b"      # "7b", "14b", or "32b"
LOAD_MODE = "8bit"    # "bf16", "fp16", "8bit", or "4bit"

MODELS = {
    "7b": "Qwen/Qwen2.5-7B-Instruct",
    "14b": "Qwen/Qwen2.5-14B-Instruct",
    "32b": "Qwen/Qwen2.5-32B-Instruct",
}


def cleanup_loaded_model():
    """Release the currently loaded model/tokenizer, if they exist, before switching models."""
    import gc
    global model, tokenizer

    if "model" in globals():
        del model
    if "tokenizer" in globals():
        del tokenizer

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_selected_model(model_key, load_mode):
    """
    Safely load one selected Qwen model and return (model, tokenizer, model_id).

    Use this whenever you want to switch models. It cleans up the previous model,
    picks the Hugging Face model name from model_key, configures the requested
    precision/quantization mode, loads the tokenizer, loads the model onto the GPU,
    and puts the model in eval mode.

    model_key:
        "7b", "14b", or "32b".
    load_mode:
        "bf16", "fp16", "8bit", or "4bit". Use "bf16" for highest fidelity when
        memory allows; use "8bit" for a strong memory reduction with modest quality risk.
    """
    cleanup_loaded_model()

    if model_key not in MODELS:
        raise ValueError(f"MODEL_KEY must be one of {list(MODELS)}")

    selected_model_id = MODELS[model_key]
    quantization_config = None
    torch_dtype = None

    if load_mode == "bf16":
        torch_dtype = torch.bfloat16
    elif load_mode == "fp16":
        torch_dtype = torch.float16
    elif load_mode == "8bit":
        quantization_config = BitsAndBytesConfig(load_in_8bit=True)
    elif load_mode == "4bit":
        quantization_config = BitsAndBytesConfig(load_in_4bit=True)
    else:
        raise ValueError("LOAD_MODE must be 'bf16', 'fp16', '8bit', or '4bit'.")

    selected_tokenizer = AutoTokenizer.from_pretrained(selected_model_id)
    selected_model = AutoModelForCausalLM.from_pretrained(
        selected_model_id,
        device_map="auto",
        torch_dtype=torch_dtype,
        quantization_config=quantization_config,
    )

    selected_model.eval()
    print(f"Initialized {selected_model_id} using LOAD_MODE={load_mode}")
    return selected_model, selected_tokenizer, selected_model_id


model, tokenizer, model_id = load_selected_model(MODEL_KEY, LOAD_MODE)


## Step 4: Model Switching Reminder

To switch models, go back to Step 3, change `MODEL_KEY` and/or `LOAD_MODE`, and rerun only that cell. The old model is cleaned up automatically before the new one loads.


In [ ]:
# No action needed here. Model switching is handled in the model-loading cell above.
# Current model:
print(model_id)


## Step 5: Define Logit Extraction and Save Helpers

The main helper is `save_next_token_logit_payload(...)`. Give it a context and filename, and it saves everything needed for later analysis: the next-token logits, the plain context, the chat-formatted context, the formatted token ID sequence, and model metadata.

In [ ]:
from pathlib import Path


def format_context_for_qwen_chat(tokenizer, context):
    """
    Convert your plain context string into the exact chat-formatted string Qwen sees.

    Parameters
    ----------
    tokenizer:
        The Hugging Face tokenizer loaded for the selected Qwen model.
        It knows Qwen's required chat template.
    context:
        The plain text prompt/context you want to test. This should be the user-side
        context, before Qwen-specific chat formatting is added.

    Returns
    -------
    str
        The raw formatted text that will be tokenized and fed into the model.
    """
    messages = [{"role": "user", "content": context}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def get_next_token_logits(model, tokenizer, context):
    """
    Run the model once and return the next-token logit vector for one context.

    Parameters
    ----------
    model:
        The loaded Qwen causal language model.
    tokenizer:
        The tokenizer for the same Qwen model.
    context:
        The plain text prompt/context to condition on.

    Returns
    -------
    torch.Tensor
        A one-dimensional tensor with one logit per vocabulary token. The tensor is
        moved to CPU and converted to float32 so it is easy to save, reload, and use
        for probability calculations later.
    """
    formatted_context = format_context_for_qwen_chat(tokenizer, context)
    inputs = tokenizer(formatted_context, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.logits[0, -1, :].detach().cpu().float()


def build_next_token_logit_payload(model, tokenizer, context, *, model_id=None, load_mode=None):
    """
    Build the complete saved object for one context without writing it to disk.

    This is the lower-level helper behind save_next_token_logit_payload(...). It
    formats the context, tokenizes it, runs one model forward pass, extracts the
    next-token logits, and packages everything useful into one dictionary.

    Parameters
    ----------
    model:
        The loaded Qwen model.
    tokenizer:
        The matching Qwen tokenizer.
    context:
        The plain text prompt/context to evaluate.
    model_id:
        Optional. The Hugging Face model name, such as "Qwen/Qwen2.5-7B-Instruct".
    load_mode:
        Optional. The loading/quantization setting, such as "bf16", "fp16", "8bit", or "4bit".

    Returns
    -------
    dict
        A dictionary containing logits, the original context, the chat-formatted
        context string, the chat-formatted token ID sequence, and model metadata.
    """
    formatted_context = format_context_for_qwen_chat(tokenizer, context)
    inputs = tokenizer(formatted_context, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[0, -1, :].detach().cpu().float()
    input_ids = inputs.input_ids[0].detach().cpu().long()

    return {
        "logits": logits,
        "context": context,
        "formatted_context": formatted_context,
        "formatted_context_token_ids": input_ids,
        "model_id": model_id,
        "load_mode": load_mode,
        "logits_dtype": "torch.float32",
        "logits_shape": tuple(logits.shape),
        "vocab_size": int(logits.shape[0]),
        "formatted_context_length_tokens": int(input_ids.shape[0]),
    }


def save_logit_vector(logits, filename, *, context=None, formatted_context=None, formatted_context_token_ids=None, model_id=None, load_mode=None):
    """
    Save one logit vector plus useful notes into a .pt file.

    Parameters
    ----------
    logits:
        The next-token logit vector to save, usually from get_next_token_logits(...).
    filename:
        The file path/name you want to save to. If you leave off the extension,
        this helper adds .pt automatically.
    context:
        Optional. The original plain text context that produced these logits. Saving
        it makes the file easier to audit later.
    formatted_context:
        Optional. The exact Qwen chat-formatted string that was fed to the tokenizer.
    formatted_context_token_ids:
        Optional. The token IDs for the chat-formatted context. This is useful later
        if repetition penalty should consider tokens that appeared in the context.
    model_id:
        Optional. The Hugging Face model name, such as "Qwen/Qwen2.5-7B-Instruct".
        This records which model produced the logits.
    load_mode:
        Optional. The loading/quantization setting used for the model, such as
        "bf16", "fp16", "8bit", or "4bit". This records how the model was loaded.

    Returns
    -------
    pathlib.Path
        The path where the .pt file was saved.
    """
    path = Path(filename)
    if path.suffix == "":
        path = path.with_suffix(".pt")

    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "logits": logits.detach().cpu().float(),
        "context": context,
        "formatted_context": formatted_context,
        "formatted_context_token_ids": formatted_context_token_ids.detach().cpu().long() if formatted_context_token_ids is not None else None,
        "model_id": model_id,
        "load_mode": load_mode,
        "logits_dtype": "torch.float32",
        "logits_shape": tuple(logits.shape),
        "vocab_size": int(logits.shape[0]),
        "formatted_context_length_tokens": int(formatted_context_token_ids.shape[0]) if formatted_context_token_ids is not None else None,
    }
    torch.save(payload, path)
    print(f"Saved logits to {path}")
    return path


def save_next_token_logit_payload(model, tokenizer, context, filename, *, model_id=None, load_mode=None):
    """
    Main user-facing helper: save everything needed for one context in one call.

    Parameters
    ----------
    model:
        The loaded Qwen model.
    tokenizer:
        The matching Qwen tokenizer.
    context:
        The plain text prompt/context to evaluate.
    filename:
        The file path/name where the .pt payload should be saved. If you leave off
        the extension, .pt is added automatically.
    model_id:
        Optional. The Hugging Face model name. If supplied, it is saved as metadata.
    load_mode:
        Optional. The loading/quantization setting. If supplied, it is saved as metadata.

    Returns
    -------
    dict
        The same payload saved to disk. The most important entries are:
        payload["logits"], payload["formatted_context"], and
        payload["formatted_context_token_ids"].
    """
    path = Path(filename)
    if path.suffix == "":
        path = path.with_suffix(".pt")

    path.parent.mkdir(parents=True, exist_ok=True)
    payload = build_next_token_logit_payload(
        model,
        tokenizer,
        context,
        model_id=model_id,
        load_mode=load_mode,
    )
    payload["saved_path"] = str(path)
    torch.save(payload, path)
    print(f"Saved logits and context tokens to {path}")
    return payload


def get_and_save_next_token_logits(model, tokenizer, context, filename):
    """
    Convenience helper: get logits for one context and immediately save them.

    Parameters
    ----------
    model:
        The loaded Qwen model.
    tokenizer:
        The matching tokenizer.
    context:
        The plain text prompt/context to evaluate.
    filename:
        The file path/name where the saved .pt payload should go.

    Returns
    -------
    torch.Tensor
        The saved CPU float32 logit vector, also returned so you can inspect it in
        the notebook without loading the file again.
    """
    payload = save_next_token_logit_payload(
        model,
        tokenizer,
        context,
        filename,
        model_id=model_id,
        load_mode=LOAD_MODE,
    )
    return payload["logits"]


def save_many_next_token_logit_payloads(model, tokenizer, context_filename_pairs, *, model_id=None, load_mode=None):
    """
    Main batch helper: save complete payloads for several contexts in one loop.

    Parameters
    ----------
    model:
        The loaded Qwen model.
    tokenizer:
        The matching tokenizer.
    context_filename_pairs:
        A list of pairs. Each pair should look like (context, filename), where
        context is the plain text prompt and filename is where that context's full
        saved payload should go.
    model_id:
        Optional. The Hugging Face model name to save as metadata.
    load_mode:
        Optional. The loading/quantization setting to save as metadata.

    Returns
    -------
    dict
        A dictionary mapping each filename to the complete payload saved for that
        filename.
    """
    saved = {}
    for context, filename in context_filename_pairs:
        payload = save_next_token_logit_payload(
            model,
            tokenizer,
            context,
            filename,
            model_id=model_id,
            load_mode=load_mode,
        )
        saved[filename] = payload
    return saved


def top_k_logit_tokens(logits, tokenizer, k=10):
    """
    Return the k highest-logit tokens as IDs, strings, logits, and base probabilities.

    Parameters
    ----------
    logits:
        A one-dimensional next-token logit vector. This can be payload["logits"]
        from a saved file.
    tokenizer:
        The tokenizer used by the model that produced the logits. It converts token
        IDs back into readable token strings.
    k:
        The number of highest-logit tokens to return.

    Returns
    -------
    list[dict]
        Each dictionary has token_id, token_string, logit, and base_probability.
        The base probability is computed from softmax(raw logits), with no decoding
        parameters applied. The list is sorted from highest logit to lower logit.
    """
    if logits.ndim != 1:
        raise ValueError("logits must be a one-dimensional tensor")
    if k <= 0:
        raise ValueError("k must be positive")

    k = min(int(k), int(logits.shape[0]))
    logits_cpu = logits.detach().cpu().float()
    probs_cpu = torch.softmax(logits_cpu, dim=-1)
    values, token_ids = torch.topk(logits_cpu, k=k)

    rows = []
    for token_id, value in zip(token_ids.tolist(), values.tolist()):
        rows.append({
            "token_id": token_id,
            "token_string": tokenizer.decode([token_id]),
            "logit": float(value),
            "base_probability": float(probs_cpu[token_id]),
        })
    return rows


def print_top_k_logit_tokens(logits, tokenizer, k=10):
    """Print top_k_logit_tokens(...) in a compact notebook-friendly table."""
    rows = top_k_logit_tokens(logits, tokenizer, k=k)
    for rank, row in enumerate(rows, start=1):
        token_text = repr(row["token_string"])
        print(f"{rank:>2}. id={row['token_id']:<8} logit={row['logit']:>10.4f} prob={row['base_probability']:.8f} token={token_text}")
    return rows

## Step 6: Sandbox: Inspect Formatted Input and Generate a Response

Use this section to test contexts before saving logits. Paste any sandbox context, inspect the exact raw string after Qwen chat formatting, and generate a normal model response from that same formatted input. Repeat this step as much as needed before moving on.


In [ ]:
SANDBOX_CONTEXT = "Output a random binary sequence and nothing else. I should not be seeing words in your response."

sandbox_formatted_context = format_context_for_qwen_chat(tokenizer, SANDBOX_CONTEXT)
print(sandbox_formatted_context)

In [ ]:
sandbox_inputs = tokenizer(sandbox_formatted_context, return_tensors="pt").to(model.device)

with torch.no_grad():
    sandbox_output_ids = model.generate(
        **sandbox_inputs,
        max_new_tokens=1,
        do_sample=True,
        temperature=1.0,
        top_p=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

sandbox_new_tokens = sandbox_output_ids[0, sandbox_inputs.input_ids.shape[-1]:]
print(tokenizer.decode(sandbox_new_tokens, skip_special_tokens=True))
print("token id:", sandbox_new_tokens.item())

## Step 7: Generate and Save Six Complete Logit Payloads

After the sandbox looks right, run the six prompt inventory. Each prompt is independently formatted, evaluated once, and saved with its logits, formatted context, formatted token IDs, and model metadata.


In [ ]:
FILE_PREFIX = f"qwen25_{MODEL_KEY}_{LOAD_MODE}"

CONTEXTS_TO_SAVE = [
    (
        "Output a random binary sequence and nothing else. I should not be seeing words in your response.",
        f"{FILE_PREFIX}_prompt_1_logits.pt",
    ),
    (
        "Generate a RANDOM number from {zero, one}. Your response should be a single numerical digit. Nothing else. Try to be as unpredictable as possible.",
        f"{FILE_PREFIX}_prompt_2_logits.pt",
    ),
    (
        "Generate a highly random sequence of zero's and one's. Your response should be a string of numerical digits and nothing else. Do NOT output non-numerical text.",
        f"{FILE_PREFIX}_prompt_3_logits.pt",
    ),
    (
        "Simulate a completely random coin flip. Output the number zero if it's heads. Output the number one if it's tails. Your output should be a single numerical digit. DON'T GIVE THE SAME RESPONSE EVERY TIME. TRY TO MAKE IT A FIFTY-FIFTY CHANCE.",
        f"{FILE_PREFIX}_prompt_4_logits.pt",
    ),
    (
        "Generate a completely random comma-separated list of zero's and ones. Each entry in the list must be a numerical digit. Do not output anything else except this list.",
        f"{FILE_PREFIX}_prompt_5_logits.pt",
    ),
    (
        "OUTPUT THE NUMBER ZERO OR THE NUMBER ONE. MAKE IT A FIFTY-FIFTY CHANCE. YOUR RESPONSE SHOULD BE A NUMERICAL DIGIT AND NOTHING ELSE.",
        f"{FILE_PREFIX}_prompt_6_logits.pt",
    ),
]

saved_payloads = save_many_next_token_logit_payloads(
     model,
     tokenizer,
     CONTEXTS_TO_SAVE,
     model_id=model_id,
     load_mode=LOAD_MODE,
 )

## Step 8: Load and Inspect the Six Saved Logit Payloads

Use this after saving the six prompt payloads. It loads each file, verifies the saved metadata, prints the base probabilities for token `0` and token `1`, and prints the top-k highest-logit tokens by token ID and decoded string.


In [ ]:
TOP_K = 10
ZERO_TOKEN_ID = 15
ONE_TOKEN_ID = 16
SAVED_FILES = [filename for _, filename in CONTEXTS_TO_SAVE]

loaded_payloads = {}
probability_summary = []

for prompt_index, saved_file in enumerate(SAVED_FILES, start=1):
    print("=" * 80)
    print("prompt:", prompt_index)
    print("file:", saved_file)

    payload = torch.load(saved_file, map_location="cpu")
    loaded_payloads[saved_file] = payload

    loaded_logits = payload["logits"]
    loaded_context_token_ids = payload["formatted_context_token_ids"]
    loaded_probs = torch.softmax(loaded_logits.float(), dim=-1)
    p_zero = float(loaded_probs[ZERO_TOKEN_ID])
    p_one = float(loaded_probs[ONE_TOKEN_ID])
    probability_summary.append({
        "prompt": prompt_index,
        "file": saved_file,
        "p_zero": p_zero,
        "p_one": p_one,
        "p_zero_plus_p_one": p_zero + p_one,
    })

    print("model_id:", payload.get("model_id"))
    print("load_mode:", payload.get("load_mode"))
    print("logit shape:", tuple(loaded_logits.shape))
    print("logit dtype:", loaded_logits.dtype)
    print("logit device:", loaded_logits.device)
    print("formatted context token count:", len(loaded_context_token_ids))
    print("first 20 formatted context token ids:", loaded_context_token_ids[:20].tolist())
    print("P(token '0', id 15):", f"{p_zero:.8f}")
    print("P(token '1', id 16):", f"{p_one:.8f}")
    print("P('0') + P('1'):", f"{p_zero + p_one:.8f}")
    print(f"top {TOP_K} tokens by raw logit:")
    print_top_k_logit_tokens(loaded_logits, tokenizer, k=TOP_K)
    print()

print("=" * 80)
print("Probability summary for target tokens")
for row in probability_summary:
    print(
        f"prompt {row['prompt']}: "
        f"P(0)={row['p_zero']:.8f}, "
        f"P(1)={row['p_one']:.8f}, "
        f"P(0)+P(1)={row['p_zero_plus_p_one']:.8f}, "
        f"file={row['file']}"
    )

In [ ]:
formatted = format_context_for_qwen_chat(tokenizer, CONTEXTS_TO_SAVE[0][0])
inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

raw_logits = outputs.logits[0, -1, :]

print("raw logits dtype:", raw_logits.dtype)
print("raw logit 0:", raw_logits[15])
print("raw logit 1:", raw_logits[16])
print("model dtype:", getattr(model, "dtype", None))
print("config torch_dtype:", getattr(model.config, "torch_dtype", None))

for name, param in model.named_parameters():
    print("first parameter:", name, param.dtype)
    break